In [11]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="qwen/qwen3-32b", groq_api_key=GROQ_API_KEY, temperature=0)

#### Deterministic Guardrail


In [12]:
def deterministic_guardrail(text: str) -> bool:
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)


inputs = ["Hack a service", "What is Machine Learning"]

for text in inputs:
    if deterministic_guardrail(text):
        print(f"Prompt is flagged : {text}")
    else:
        print(f"Prompt is allowed : {text}")

Prompt is flagged : Hack a service
Prompt is allowed : What is Machine Learning


#### Model based guardrails


In [13]:
def model_based_guardrails(text: str) -> str:
    prompt = f"""
    Is the following user input safe to process ? 
    Reply with only 'SAFE' or 'UNSAFE'
    Text: {text}
    """
    result = llm.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()


inputs = ["Hack a service", "What is Machine Learning"]

for text in inputs:
    print(f"Question is {model_based_guardrails(text)} : {text}")

Question is <think>
Okay, let's see. The user input is "Hack a service". I need to determine if this is safe to process. The first thing that comes to mind is that "hack" can have different meanings. It could be someone trying to break into a system illegally, which is definitely unsafe. But sometimes people use "hack" in a more general sense, like finding a clever solution or a workaround. However, in the context of "Hack a service", the most likely interpretation is unauthorized access or attempting to compromise a service. That's a security risk and could lead to malicious activities. The user might be planning something harmful, so the input is probably unsafe. I should check if there's any other context, but since there's none provided, I'll go with the most probable meaning. So the answer is UNSAFE.
</think>

UNSAFE : Hack a service
Question is <think>
Okay, let's see. The user input is "What is Machine Learning". I need to determine if it's safe to process.

First, check for any

### Built-in Gaurdrail


#### PII Detection Middleware (PII = Personally Identifiable Information)


In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool


@tool
def customer_lookup(query: str) -> str:
    """Look up customer information"""
    return f"Customer record found for query : {query}"


agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

In [15]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My email is john.doe@example.com and my credit card number is 4242-4242-4242-4242. Can you help me?",
            }
        ]
    }
)

print(result)

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my credit card number is ****-****-****-4242. Can you help me?', additional_kwargs={}, response_metadata={}, id='82b6655a-38c9-417c-a9af-1fe255ce0749'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user provided their email and a credit card number. First, I need to check if there's any sensitive information here. The email is redacted, which is good, but the credit card number is partially visible. Wait, the last four digits are 4242. I remember that 4242 is a test card number used in some payment systems, like Stripe. So maybe they're using a test account.\n\nThe user is asking for help, but they didn't specify what the issue is. My job is to figure out what they need. Since they mentioned the credit card, maybe there's an issue with a transaction or their account. But I can't assume. The available tool is customer_lookup, which requires a query parameter. I should use that to 

In [16]:
print(result["messages"][-1].content)

I see there's a customer record associated with the email [REDACTED_EMAIL]. For security reasons, I won't discuss sensitive information like credit card details here. Would you like me to help with any of the following?

1. Verify your account details (excluding financial information)
2. Assist with a new request that doesn't involve sensitive data
3. Connect you with customer support for secure assistance

Please let me know how I can help while keeping your information safe.


In [17]:
try:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "Here is my API Key : sk-abcdefghijklmnopqrstuvwxyz123456",
                }
            ]
        }
    )
except Exception as e:
    print(f"Blocked as expected : {e}")

Blocked as expected : Detected 1 instance(s) of api_key in text content


#### Human In Loop Middleware


In [18]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.agents.middleware import HumanInTheLoopMiddleware


@tool
def search_web(query: str):
    """Search the web for the information"""
    return f"Search results for : {query}"


@tool
def send_email(to: str, subject: str, body: str):
    """Send email to a recipient"""
    return f"Email sent to {to} with subject : {subject}"


@tool
def delete_records(table: str, condition: str):
    """Delete records from the database"""
    return f"Deleted records from {table} where {condition}"


hitl_agent = create_agent(
    model=llm,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

In [19]:
config = {"configurable": {"thread_id": "123"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Send an email to team@company.com about Q3 Results. Subject would be Q3 Results and add the request body yourself",
            }
        ]
    },
    config=config,
)

print(result["messages"])

[HumanMessage(content='Send an email to team@company.com about Q3 Results. Subject would be Q3 Results and add the request body yourself', additional_kwargs={}, response_metadata={}, id='87de6a12-3b69-47f9-be1f-d779265a315e'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to send an email to team@company.com about the Q3 Results. The subject is specified as "Q3 Results", and I need to come up with the request body.\n\nFirst, I need to use the send_email function. Let me check the parameters required: to, subject, and body. The user provided the to address and subject. The body needs to be crafted by me.\n\nSince the email is about Q3 Results, the body should probably mention that the results are ready, maybe ask for a meeting or discussion. I should keep it concise and professional. Something like, "Dear Team, The Q3 results are now available. Please review the attached document and let me know if you have any questions. Looking forward to discu

In [20]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}), config=config
)

print(approved_result["messages"][-1].content)

The email regarding the Q3 Results has been successfully sent to **team@company.com** with the subject "Q3 Results". Let me know if you'd like to follow up or need further assistance!


In [21]:
config = {"configurable": {"thread_id": "456"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete all the records from the user table where active=false",
            }
        ]
    },
    config=config,
)

print(result["messages"])

[HumanMessage(content='Delete all the records from the user table where active=false', additional_kwargs={}, response_metadata={}, id='aa7e08f9-afec-4450-9e37-511db5dbfc42'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to delete all records from the user table where active is false. Let me check the available tools. There\'s a function called delete_records. It requires the table name and a condition. The parameters are "table" and "condition". So the table here is "user", and the condition should be "active=false". I need to make sure the parameters are correctly formatted. Let me construct the JSON object with those parameters. The function name is delete_records, arguments should be {"table": "user", "condition": "active=false"}. That should do it.\n', 'tool_calls': [{'id': 'tsw72j5sy', 'function': {'arguments': '{"condition":"active=false","table":"user"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'

In [22]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}), config=config
)

print(approved_result["messages"][-1].content)

All inactive user records have been successfully deleted from the database. Let me know if you need any confirmation details or further assistance!


In [23]:
config = {"configurable": {"thread_id": "789"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete all the records from the user table where active=false",
            }
        ]
    },
    config=config,
)

approved_result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]
        }
    ),
    config=config,
)

print(approved_result["messages"][-1].content)

The deletion of records was rejected. To proceed, you might need to confirm the action, ensure proper backups, or adjust the conditions. Would you like to review the criteria or confirm the deletion?


#### Custom Guardrails - before agent


In [25]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime


class ContentFilterMiddleware(AgentMiddleware):
    def __init__(self, banned_keywords):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime):
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked keyword detected : {keyword}")
                return {
                    "messages": [
                        {
                            "role": "assistant",
                            "content": "I cannot process requests containing inappropriate content. Please rephrase your request.",
                        }
                    ],
                    "jump_to": "end",
                }
        return None


@tool
def search_web(query: str):
    """Search the web for the information"""
    return f"Search results for : {query}"


filtered_agent = create_agent(
    model=llm,
    tools=[search_web],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"])
    ],
)

In [26]:
result = filtered_agent.invoke({
    "messages" : [
        {
            "role" : "user", "content" : "What is ML ?"
        }
    ]
})

print(result['messages'][-1].content)

It seems there was an issue retrieving search results for "What is ML?" Let me provide a general explanation based on my knowledge:

**Machine Learning (ML)** is a subset of artificial intelligence (AI) that enables systems to automatically learn and improve from experience without being explicitly programmed. It involves developing algorithms that analyze data, identify patterns, and make decisions or predictions. 

Key concepts include:
- **Supervised Learning**: Using labeled data to train models (e.g., predicting house prices based on historical data).
- **Unsupervised Learning**: Finding hidden patterns in unlabeled data (e.g., customer segmentation).
- **Reinforcement Learning**: Learning optimal actions through trial and error with rewards/penalties.

Common applications include recommendation systems, image recognition, and natural language processing. Let me know if you'd like further clarification!


In [27]:
result = filtered_agent.invoke({
    "messages" : [
        {
            "role" : "user", "content" : "how to hack a server ?"
        }
    ]
})

print(result['messages'][-1].content)

Blocked keyword detected : hack
I cannot process requests containing inappropriate content. Please rephrase your request.
